In [ ]:
import pandas as pd

df = pd.read_csv("../out.csv")

df

,DataHora,Municipio,Bioma,DiaSemChuva,Precipitacao,RiscoFogo,Fogo
0,2025-08,TAMBAÚ,2.480000,9.600000,2.784000,0.808000,1
1,2025-08,OLÍMPIA,2.566667,12.833333,0.016667,0.998333,1
2,2026-03,ÁGUAS DE LINDÓIA,1.700000,0.000000,0.000000,0.030000,1
3,2025-07,GUAPIARA,1.700000,13.625000,0.012500,0.395000,1
4,2025-07,IBIÚNA,1.700000,15.882353,0.005882,0.667647,1
...,...,...,...,...,...,...,...
2088,2026-02,NUPORANGA,3.000000,23.840036,21.600000,0.176805,0
2089,2026-03,NUPORANGA,3.000000,77.449247,0.000000,0.477131,0
2090,2026-04,NUPORANGA,3.000000,84.657646,3.300000,0.348720,0
2091,2026-05,NUPORANGA,3.000000,103.052108,0.000000,0.121431,0


In [20]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[["Bioma", "DiaSemChuva", "Precipitacao", "RiscoFogo"]]
y = df["Fogo"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [22]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(
    hidden_layer_sizes=(8),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42
)

mlp.fit(X_train_scaled, y_train)

y_pred = mlp.predict(X_test_scaled)
print(f"Acurácia: {accuracy_score(y_test, y_pred):.4f}\n")
print("Matriz de Confusão:")
print(confusion_matrix(y_test, y_pred))
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred))

Acurácia: 0.9928

Matriz de Confusão:
[[381   0]
 [  3  35]]

Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       381
           1       1.00      0.92      0.96        38

    accuracy                           0.99       419
   macro avg       1.00      0.96      0.98       419
weighted avg       0.99      0.99      0.99       419



In [23]:
from sklearn.gaussian_process.kernels import RBF
from sklearn.model_selection import train_test_split
from sklearn.gaussian_process import GaussianProcessClassifier

kernel = 1.0 * RBF(length_scale=1.0)

gpc = GaussianProcessClassifier(
    kernel=kernel,
    random_state=42,
    n_jobs=-1  # Utiliza todos os núcleos do processador para acelerar
)

gpc.fit(X_train_scaled, y_train)

y_pred = gpc.predict(X_test_scaled)
y_prob = gpc.predict_proba(X_test_scaled)

print(f"Acurácia: {accuracy_score(y_test, y_pred):.4f}\n")
print("Matriz de Confusão:")
print(confusion_matrix(y_test, y_pred))
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred))

/home/erick/Documentos/queimadas/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Acurácia: 0.9857

Matriz de Confusão:
[[378   3]
 [  3  35]]

Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       381
           1       0.92      0.92      0.92        38

    accuracy                           0.99       419
   macro avg       0.96      0.96      0.96       419
weighted avg       0.99      0.99      0.99       419



In [61]:
teste = pd.DataFrame([
    {
        "Bioma": 3,
        "DiaSemChuva": 60,
        "Precipitacao": 0,
        "RiscoFogo": 1
    }
])

teste_s = scaler.transform(teste)

print(f'Probabilidade de incêndio (MLP): {mlp.predict_proba(teste_s)[0][1]:.2f}')
print(f'Probabilidade de incêndio (GPC): {gpc.predict_proba(teste_s)[0][1]:.2f}')

Probabilidade de incêndio (MLP): 0.62
Probabilidade de incêndio (GPC): 0.68
